# Normalization for Identity Resolution Phase: Kaggle

This notebook normalizes the Kaggle airport dataset (`kaggleMapped.xml`) for the identity resolution phase.

Steps applied:
1. Lowercase & remove punctuation
2. Strip spaces
3. Unicode normalization (replace ï, ö, ä, accented chars)
4. Translation *(SKIPPED – pending coaching session)*
5. Abbreviation expansion
6. Remove stopwords and common airport words
7. Country columns *(handled separately via pycountry)*

In [31]:
import re
import unicodedata
from pathlib import Path

import pandas as pd
from PyDI.io import load_xml

DATA_DIR = Path("../../schema/kaggle")
OUTPUT_DIR = Path("../../data/normalized_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

XML_FILE = DATA_DIR / "kaggleMapped.xml"

df = load_xml(XML_FILE, name="kaggle")
df.attrs['source'] = 'kaggle'

print(f"Loaded {len(df)} records")
print(f"Columns: {list(df.columns)}")
df.head(3)

Loaded 6391 records
Columns: ['id', 'name', 'iata', 'icao', 'lat', 'lon', 'countryCode', 'countryCodeA3', 'countryName', 'city', 'cityIata', 'timezone']


,id,name,iata,icao,lat,lon,countryCode,countryCodeA3,countryName,city,cityIata,timezone
0,kg_1,Pilanesberg Intl,NTY,FAPN,-25.333822,27.173358,ZA,ZAF,South Africa,Sun City,NTY,Africa/Johannesburg
1,kg_2,Clovis Muni,CVN,KCVN,34.425139,-103.079278,US,USA,United States of America,"Clovis, New Mexico",CVN,America/Denver
2,kg_3,Cannon Afb,CVS,KCVS,34.382775,-103.322147,US,USA,United States of America,"Clovis, New Mexico",CVN,America/Denver


## 1. Lowercase & remove punctuation for string columns

Targeted columns: `name`, `countryName`, `city`

Removes: commas, dots, parentheses, `'`, `"`, hyphens (kept as space), and other punctuation.

In [32]:
STRING_COLS = ["name", "countryName", "city"]

# Characters to remove completely
REMOVE_CHARS = re.compile(r"[,\.\(\)'\"\!\?\:\;\`\~]")  
# Hyphens/slashes -> space so 'Saint-Pierre' becomes 'saint pierre'
HYPHEN_CHARS = re.compile(r"[-/]")  

def lowercase_remove_punct(text):
    if not isinstance(text, str):
        return text
    text = text.lower()
    text = HYPHEN_CHARS.sub(" ", text)
    text = REMOVE_CHARS.sub("", text)
    return text

for col in STRING_COLS:
    if col in df.columns:
        df[col] = df[col].apply(lowercase_remove_punct)

print("Sample after step 1:")
df[["id", "name", "countryName", "city"]].head(10)

Sample after step 1:


,id,name,countryName,city
0,kg_1,pilanesberg intl,south africa,sun city
1,kg_2,clovis muni,united states of america,clovis new mexico
2,kg_3,cannon afb,united states of america,clovis new mexico
3,kg_4,scammon bay airport,united states of america,scammon bay alaska
4,kg_5,kapit,malaysia,kapit
5,kg_6,eagle river,united states of america,eagle river wisconsin
6,kg_7,darnley island airport,australia,darnley island
7,kg_8,lourdes de blanc sablon airport,canada,blanc sablon
8,kg_9,uzhhorod international airport,ukraine,uzhgorod
9,kg_10,andahuaylas,peru,andahuaylas


## 2. Strip spaces

Remove leading/trailing whitespace and collapse multiple internal spaces.

In [33]:
MULTI_SPACE = re.compile(r"\s+")

def strip_spaces(text):
    if not isinstance(text, str):
        return text
    return MULTI_SPACE.sub(" ", text).strip()

for col in STRING_COLS:
    if col in df.columns:
        df[col] = df[col].apply(strip_spaces)

# Show entries that had trailing spaces (like kg_5 "Kapit               ")
print("Sample after step 2:")
df[["id", "name"]].head(10)

Sample after step 2:


,id,name
0,kg_1,pilanesberg intl
1,kg_2,clovis muni
2,kg_3,cannon afb
3,kg_4,scammon bay airport
4,kg_5,kapit
5,kg_6,eagle river
6,kg_7,darnley island airport
7,kg_8,lourdes de blanc sablon airport
8,kg_9,uzhhorod international airport
9,kg_10,andahuaylas


## 3. Unicode normalization

Uses NFKD decomposition to separate base characters from diacritics, then encodes to ASCII (ignoring the diacritic combining characters).

Examples:
- ï, ö, ä → i, o, a
- São Paulo → sao paulo
- Piñon → pinon
- Çarşamba → carsamba

In [34]:
def normalize_unicode(text):
    if not isinstance(text, str):
        return text
    # NFKD decomposes combined chars; encode/decode drops combining diacritics
    return unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")

for col in STRING_COLS:
    if col in df.columns:
        df[col] = df[col].apply(normalize_unicode)

# Verify: show entries that previously had diacritics
test_cases = [
    normalize_unicode("São Paulo"),
    normalize_unicode("Piñon"),
    normalize_unicode("Çarşamba"),
    normalize_unicode("Düsseldorf"),
    normalize_unicode("München"),
]
for original, result in zip(
    ["São Paulo", "Piñon", "Çarşamba", "Düsseldorf", "München"], test_cases
):
    print(f"  {original!r:25s} -> {result!r}")

print("\nSample after step 3:")
df[["id", "name", "countryName"]].head(10)

  'São Paulo'               -> 'Sao Paulo'
  'Piñon'                   -> 'Pinon'
  'Çarşamba'                -> 'Carsamba'
  'Düsseldorf'              -> 'Dusseldorf'
  'München'                 -> 'Munchen'

Sample after step 3:


,id,name,countryName
0,kg_1,pilanesberg intl,south africa
1,kg_2,clovis muni,united states of america
2,kg_3,cannon afb,united states of america
3,kg_4,scammon bay airport,united states of america
4,kg_5,kapit,malaysia
5,kg_6,eagle river,united states of america
6,kg_7,darnley island airport,australia
7,kg_8,lourdes de blanc sablon airport,canada
8,kg_9,uzhhorod international airport,ukraine
9,kg_10,andahuaylas,peru


## 4. Translation *(SKIPPED FOR NOW: pending coaching session)*

OurAirports has thousands of non-English airport names. Discuss with professor whether translation is worthwhile.

## 5. Abbreviation expansion

The Kaggle dataset uses various abbreviations in the `name` column. The abbreviations below were identified by:
1. The known list from the task description
2. A frequency analysis of title-case and all-caps short tokens in the name column

Expansion is applied as **whole-word** substitution (word boundaries `\b`) to avoid accidental replacements inside longer words.

In [35]:
# Abbreviation map (lowercase key -> expanded form, also lowercase)
# Order matters for overlapping patterns – more specific first
ABBREVIATION_MAP = [
    # Directly from task description
    (r"\bintl\b",  "international"),
    (r"\brgnl\b",  "regional"),
    (r"\bmuni\b",  "municipal"),
    (r"\bafb\b",   "air force base"),
    (r"\bfld\b",   "field"),
    # st: keep as 'saint' in airport context ("St. Louis" -> saint louis)
    # street is much less common in airport names than saint
    (r"\bst\.?\b", "saint"),
    # Additional abbreviations found via frequency analysis
    (r"\baaf\b",   "army air field"),
    (r"\bnas\b",   "naval air station"),
    (r"\bab\b",    "air base"),
    (r"\bstn\b",   "station"),
    (r"\bco\.?\b", "county"),
    (r"\bmem\b",   "memorial"),
    (r"\blrrs\b",  "long range radar site"),
    (r"\bhwy\b",   "highway"),
    (r"\bmt\.?\b", "mount"),
    (r"\bft\.?\b", "fort"),
    (r"\bpt\.?\b", "point"),
    (r"\bjct\.?\b","junction"),
    (r"\bblvd\.?\b","boulevard"),
]

# Compile patterns once
COMPILED_ABBREVS = [
    (re.compile(pattern, re.IGNORECASE), expansion)
    for pattern, expansion in ABBREVIATION_MAP
]

def expand_abbreviations(text):
    if not isinstance(text, str):
        return text
    for pattern, expansion in COMPILED_ABBREVS:
        text = pattern.sub(expansion, text)
    return text

# Only apply to 'name' column (abbreviations appear there, not in country/city)
df["name"] = df["name"].apply(expand_abbreviations)

# Strip spaces again after expansion (expansions may create multiple spaces)
df["name"] = df["name"].apply(strip_spaces)

# Verify some examples
examples = [
    "pilanesberg intl",
    "clovis muni",
    "cannon afb",
    "easterwood fld",
    "tallahassee rgnl",
    "jacksonville nas",
    "yokota ab",
    "ladd aaf",
    "imperial co",
    "vologda rail stn",
]
print("Abbreviation expansion examples:")
for ex in examples:
    print(f"  {ex!r:40s} -> {expand_abbreviations(ex)!r}")

print("\nSample after step 5:")
df[["id", "name"]].head(15)

Abbreviation expansion examples:
  'pilanesberg intl'                       -> 'pilanesberg international'
  'clovis muni'                            -> 'clovis municipal'
  'cannon afb'                             -> 'cannon air force base'
  'easterwood fld'                         -> 'easterwood field'
  'tallahassee rgnl'                       -> 'tallahassee regional'
  'jacksonville nas'                       -> 'jacksonville naval air station'
  'yokota ab'                              -> 'yokota air base'
  'ladd aaf'                               -> 'ladd army air field'
  'imperial co'                            -> 'imperial county'
  'vologda rail stn'                       -> 'vologda rail station'

Sample after step 5:


,id,name
0,kg_1,pilanesberg international
1,kg_2,clovis municipal
2,kg_3,cannon air force base
3,kg_4,scammon bay airport
4,kg_5,kapit
5,kg_6,eagle river
6,kg_7,darnley island airport
7,kg_8,lourdes de blanc sablon airport
8,kg_9,uzhhorod international airport
9,kg_10,andahuaylas


## 6. Remove stopwords and common words

After abbreviation expansion many names still contain common words like `international`, `airport`, `the`, `of`, etc. that add noise to string comparators.

Strategy: remove a fixed set of domain-specific stopwords from the `name` column.

In [36]:
# Common English function words (equivalent to sklearn's ENGLISH_STOP_WORDS subset)
# Defined inline to avoid importing sklearn/scipy (binary incompatibility on some macOS builds)
ENGLISH_STOP_WORDS = frozenset([
    "a", "an", "the", "and", "or", "but", "if", "in", "on", "at", "to",
    "of", "for", "by", "with", "as", "is", "it", "its", "be", "are", "was",
    "were", "been", "have", "has", "had", "do", "does", "did", "not", "no",
    "nor", "so", "yet", "both", "either", "neither", "than", "that", "this",
    "these", "those", "which", "who", "whom", "when", "where", "how", "all",
    "each", "every", "any", "some", "such", "from", "into", "through",
    "during", "before", "after", "above", "below", "between", "out", "off",
    "over", "under", "again", "further", "then", "once", "there", "here",
])

# Extended with Romance/Germanic articles & prepositions
STOPWORDS = ENGLISH_STOP_WORDS | {
    "de", "la", "le", "el", "al", "del", "des", "les", "du", "da", "di", "der", "das",
}

STOPWORD_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(w) for w in sorted(STOPWORDS, key=len, reverse=True)) + r")\b"
)

def remove_stopwords(text):
    if not isinstance(text, str):
        return text
    text = STOPWORD_PATTERN.sub("", text)
    return strip_spaces(text)

# Keep original name before stopword removal for inspection
df["name_before_sw"] = df["name"].copy()
df["name"] = df["name"].apply(remove_stopwords)

changed = df[df["name"] != df["name_before_sw"]][["id", "name_before_sw", "name"]].head(20)
print(f"Stopword removal changed {len(df[df['name'] != df['name_before_sw']])} records")
display(changed)

df.drop(columns=["name_before_sw"], inplace=True)

Stopword removal changed 265 records


,id,name_before_sw,name
7,kg_8,lourdes de blanc sablon airport,lourdes blanc sablon airport
55,kg_56,valle del fuerte international,valle fuerte international
67,kg_68,mocimboa da praia,mocimboa praia
98,kg_99,isles of scilly,isles scilly
106,kg_107,la chinita international,chinita international
124,kg_125,bergamo orio al serio,bergamo orio serio
148,kg_149,al udeid air base,udeid air base
205,kg_206,val de loire,val loire
206,kg_207,saint pierre des corps,saint pierre corps
234,kg_235,general leite de castro airport,general leite castro airport


## 7. Country columns

`countryCode` (ISO alpha-2) is already clean and will serve as the blocking key for identity resolution.
`countryName` was normalized in steps 1–3.

In [37]:
# countryCode is already clean (ISO alpha-2) and will be used as blocking key.
# countryName was normalized in steps 1-3 (lowercase, unicode, strip).
# No further processing needed here for Kaggle.
#
# pycountry will be used in the Wikidata normalization notebook
# to derive countryCode from countryName (Wikidata has no countryCode).

print("countryCode sample:")
print(df["countryCode"].value_counts().head(10))

countryCode sample:
countryCode
US    1323
CA     371
AU     267
RU     251
CN     227
BR     187
GB     154
ID     153
FR     153
DE     127
Name: count, dtype: int64


## Final: Save normalized dataset

In [38]:
output_csv = OUTPUT_DIR / "kaggleNormalized.csv"
df.to_csv(output_csv, index=False)
print(f"Saved {len(df)} normalized records to {output_csv}")

# Summary stats
print(f"\nDataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df[["id", "name", "countryCode", "countryName", "city"]].head(15)

Saved 6391 normalized records to ../../data/normalized_outputs/kaggleNormalized.csv

Dataset shape: (6391, 12)
Columns: ['id', 'name', 'iata', 'icao', 'lat', 'lon', 'countryCode', 'countryCodeA3', 'countryName', 'city', 'cityIata', 'timezone']


,id,name,countryCode,countryName,city
0,kg_1,pilanesberg international,ZA,south africa,sun city
1,kg_2,clovis municipal,US,united states of america,clovis new mexico
2,kg_3,cannon air force base,US,united states of america,clovis new mexico
3,kg_4,scammon bay airport,US,united states of america,scammon bay alaska
4,kg_5,kapit,MY,malaysia,kapit
5,kg_6,eagle river,US,united states of america,eagle river wisconsin
6,kg_7,darnley island airport,AU,australia,darnley island
7,kg_8,lourdes blanc sablon airport,CA,canada,blanc sablon
8,kg_9,uzhhorod international airport,UA,ukraine,uzhgorod
9,kg_10,andahuaylas,PE,peru,andahuaylas
